# TCGA cohort download from the GDC API

## Overview

This notebook documents how TCGA RNA-seq count matrices and clinical metadata were downloaded from the NCI Genomic Data Commons using the helper functions in `src/TCGA_load.py`. It is retained for auditability; for routine use, the command-line call shown below is the preferred entry point.

Raw downloaded files are large and are not committed to the repository. The processed outputs expected by the cohort-analysis notebooks are written under `data/<TCGA-project>/treated_files/` as `counts.txt`, `genes.txt`, and `metadata.txt`.


## Command-line use

From the repository root, run for one or several TCGA projects:

```bash
python src/TCGA_load.py TCGA-BRCA
python src/TCGA_load.py TCGA-BRCA,TCGA-OV,TCGA-PRAD
```

By default, intermediate downloaded files are written to `data/<TCGA-project>/download_data/`, and processed files are written to `data/<TCGA-project>/treated_files/`.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

from src import TCGA_load


In [ ]:
project_list_name = ["TCGA-BRCA"]

for project_name in project_list_name:
    print(project_name)
    path_to_load = repo_root / "data" / project_name / "treated_files"
    path_to_download = repo_root / "data" / project_name / "download_data"
    path_to_load.mkdir(parents=True, exist_ok=True)
    path_to_download.mkdir(parents=True, exist_ok=True)

    rna_seq_files = TCGA_load.get_rnaseq_data(project_name)
    print(f"Number of RNA-seq files found: {len(rna_seq_files)}")

    combined_counts, sample_metadata = TCGA_load.get_counts_data(
        rna_seq_files,
        path_to_download=str(path_to_download) + "/",
    )
    TCGA_load.download_clinical_data(project_name, str(path_to_download) + "/")
    clinical_metadata = TCGA_load.load_clinical_data(
        project_name,
        sample_metadata,
        str(path_to_load) + "/",
        str(path_to_download) + "/",
    )
    TCGA_load.load_counts_data(combined_counts, clinical_metadata, str(path_to_load) + "/")


## Notes

The GDC API can occasionally return transient HTTP or connection errors. If this happens, rerun the same command: already downloaded files are skipped when their filenames are present in the download directory.
